# Desenvolvimento de Modelo de PD (Probability of Default)

## Estruturação da Base e Definição de Target
- **Definição da população elegível (`t = 0`):** Filtrei a base (`Dados_PD.csv`) na data de observação (`safra_obs`) para manter apenas a carteira ativa e adimplente, removendo contratos cancelados ou que já estavam em default (mais de 90 dias de atraso) antes da foto inicial.
- **Padronização da janela de performance:** Fixei o horizonte de acompanhamento em **12 meses** após a `safra_obs`, seguindo o padrão regulatório de Basileia e IFRS 9 — tiro uma foto do cliente hoje e observo seu comportamento ao longo de um ano.
- **Construção da variável-alvo (`default_12m`):** Marquei como `1` (Mau) todo cliente que atingiu atraso superior a 90 dias em qualquer momento dentro da janela de 12 meses, e como `0` (Bom) quem atravessou o período sem entrar em default.
- **Tratamento de contratos indeterminados (produção):** Para cenários reais onde o cliente encerra o contrato antes dos 12 meses sem ter atrasado, estabeleci a regra de mantê-lo como `0` (Bom) caso tenha completado ao menos 6 meses de observação, ou removê-lo do treino para não gerar ruído no modelo.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('Dados.csv')
df

,safra_obs,renda_mensal,limite,saldo,variacao_saldo_3m,qtd_atrasos_curtos_6m,pagou_minimo_ult_fatura,saldo_default,default_12m
0,2023,4772.63,7437.41,3496.18,0.2938,2,1,0.0,0
1,2025,4322.33,9975.62,5334.88,0.2238,0,0,0.0,0
2,2024,1469.32,3336.31,635.10,0.0475,1,1,0.0,0
3,2024,3107.31,5242.38,862.92,-0.0462,1,0,0.0,0
4,2022,6515.44,15243.24,7553.81,0.0172,0,1,0.0,0
...,...,...,...,...,...,...,...,...,...
59995,2025,3905.45,6528.77,5694.86,-0.2292,0,0,0.0,0
59996,2023,10256.45,24346.27,2830.30,0.4790,1,1,0.0,0
59997,2023,7722.87,28948.36,13178.49,0.6570,0,1,0.0,0
59998,2023,4797.51,7770.95,5863.42,0.4218,1,0,0.0,0


## Engenharia de Atributos e Split Out-of-Time (OOT)
- **Criação de variáveis e tratamento de outliers:** Criei a razão `percentual_uso` (`saldo / limite`) para capturar o nível de alavancagem do cliente e apliquei *winsorization* em `variacao_saldo_3m` (limitando entre `-0.5` e `1.0`) para evitar que valores extremos distorçam os coeficientes do modelo.
- **Separação Out-of-Time (OOT):** Dividi a base respeitando a ordem cronológica (`safra_obs < 2025` para treino e `safra_obs == 2025` para teste), simulando o comportamento real do modelo em produção ao prever o futuro com base no passado.
- **Pré-seleção de candidatas:** Defini 6 variáveis candidatas removendo redundâncias colineares (como manter `saldo` isolado junto ao `percentual_uso`).

In [ ]:
df['percentual_uso'] = df['saldo'] / df['limite']
df['variacao_saldo_clip'] = np.clip(df['variacao_saldo_3m'], -0.5, 1.0)

treino = df[df['safra_obs'] < 2025].copy()
teste = df[df['safra_obs'] == 2025].copy()


candidatas = [
    'qtd_atrasos_curtos_6m',
    'percentual_uso',
    'variacao_saldo_clip',
    'pagou_minimo_ult_fatura',
    'renda_mensal',
    'limite'
]

## Seleção de Variáveis (Information Value - IV)
- **Poder preditivo sem vazamento:** Calculei o *Information Value (IV)* exclusivamente na base de **treino** para medir a capacidade individual de cada variável em separar Bons e Maus pagadores sem gerar *data leakage*.
- **Critério de corte:** Apliquei a regra padrão de mercado (`IV >= 0.02`), selecionando as 4 variáveis mais relevantes (`qtd_atrasos_curtos_6m`, `percentual_uso`, `variacao_saldo_clip` e `pagou_minimo_ult_fatura`) e descartando `renda_mensal` e `limite` por baixo poder discriminatório.

In [ ]:
def calcular_iv(dados, feature, target='default_12m', bins=5):
    aux = dados[[feature, target]].copy()
    aux['faixa'] = pd.qcut(aux[feature], q=bins, duplicates='drop') if aux[feature].nunique() > bins else aux[feature]
    tabela = aux.groupby('faixa', observed=False)[target].agg(total='count', maus='sum')
    tabela['bons'] = tabela['total'] - tabela['maus']
    dist_maus = (tabela['maus'] + 0.5) / (tabela['maus'].sum() + 0.5)
    dist_bons = (tabela['bons'] + 0.5) / (tabela['bons'].sum() + 0.5)
    return round(((dist_maus - dist_bons) * np.log(dist_maus / dist_bons)).sum(), 4)

tabela_iv = pd.DataFrame({
    'feature': candidatas,
    'IV': [calcular_iv(treino, col) for col in candidatas]
}).sort_values(by='IV', ascending=False)

features_selecionadas = tabela_iv[tabela_iv['IV'] >= 0.02]['feature'].tolist()
print("Features selecionadas:", features_selecionadas)

Features selecionadas: ['qtd_atrasos_curtos_6m', 'percentual_uso', 'variacao_saldo_clip', 'pagou_minimo_ult_fatura']


## Treinamento do Modelo de PD (Regressão Logística)
- **Pipeline de modelagem:** Construí um pipeline combinando `StandardScaler` (para colocar todas as features na mesma escala) e `LogisticRegression` — algoritmo escolhido pela alta interpretabilidade.
- **Escore de probabilidade:** Treinei o modelo nas 4 variáveis selecionadas e gerei a probabilidade estimada de inadimplência (`pd_estimada`) para as bases de treino, teste e carteira completa.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve


modelo_pd = make_pipeline(StandardScaler(), LogisticRegression())
modelo_pd.fit(treino[features_selecionadas], treino['default_12m'])


treino['pd_estimada'] = modelo_pd.predict_proba(treino[features_selecionadas])[:, 1]
teste['pd_estimada'] = modelo_pd.predict_proba(teste[features_selecionadas])[:, 1]
df['pd_estimada'] = modelo_pd.predict_proba(df[features_selecionadas])[:, 1]

## Validação Out-of-Time e Estabilidade
- **Poder de discriminação (2025):** Avaliei a capacidade do modelo de ordenar corretamente o risco na safra *Out-of-Time* utilizando **AUC**, **Gini** e **KS**.
- **Estabilidade populacional (PSI):** Calculei o *Population Stability Index* comparando a distribuição da `pd_estimada` entre o treino e o teste (`2025`) para verificar se houve mudança de perfil da carteira (*drift*).

In [ ]:
auc = roc_auc_score(teste['default_12m'], teste['pd_estimada'])
gini = 2 * auc - 1
fpr, tpr, _ = roc_curve(teste['default_12m'], teste['pd_estimada'])
ks = np.max(tpr - fpr)

def calcular_psi(esperado, atual, bins=10):
    cortes = np.quantile(esperado, np.linspace(0, 1, bins + 1))
    cortes[0], cortes[-1] = -np.inf, np.inf
    cortes = np.unique(cortes)
    esp_pct = np.clip(np.histogram(esperado, bins=cortes)[0] / len(esperado), 1e-6, None)
    atu_pct = np.clip(np.histogram(atual, bins=cortes)[0] / len(atual), 1e-6, None)
    return np.sum((atu_pct - esp_pct) * np.log(atu_pct / esp_pct))

psi = calcular_psi(treino['pd_estimada'], teste['pd_estimada'])

print("\n--- Avaliação do Modelo de PD (Out-of-Time 2025) ---")
print(f"AUC:  {auc:.4f}  | Saudável: >= 0.70 (Ideal: > 0.75)")
print(f"Gini: {gini:.4f}  | Saudável: >= 0.40 (Ideal: > 0.50)")
print(f"KS:   {ks:.4f}  | Saudável: >= 0.30 (Ideal: > 0.40)")
print(f"PSI:  {psi:.4f}  | Saudável: <  0.10 (Alerta: 0.10 a 0.25 | Crítico: > 0.25)")


--- Avaliação do Modelo de PD (Out-of-Time 2025) ---
AUC:  0.7073  | Saudável: >= 0.70 (Ideal: > 0.75)
Gini: 0.4146  | Saudável: >= 0.40 (Ideal: > 0.50)
KS:   0.3045  | Saudável: >= 0.30 (Ideal: > 0.40)
PSI:  0.0009  | Saudável: <  0.10 (Alerta: 0.10 a 0.25 | Crítico: > 0.25)


### Leitura dos Resultados (OOT 2025)
- **Discriminação saudável:** O modelo atingiu **AUC de 0,7073**, **Gini de 0,4146** e **KS de 0,3045**, superando os patamares mínimos saudáveis de mercado para todas as métricas.
- **Alta estabilidade:** O **PSI de 0,0009** (muito abaixo do limite de `0,10`) confirma que a distribuição das probabilidades previstas permaneceu estável na safra de 2025, sem sinais de degradação ou *drift*.